# Ponto de Controle
Este notebook valida e escreve dados transformados no Google Sheets.

In [ ]:
from extract import read_df
from treat.utils.write_dataframe_to_sheet import write_dataframe_to_sheet
from treat.utils.datas import normalize_date_to_str_DD_M_YYYY
import pandas as pd
import os
import datetime as dt

In [ ]:
DRY_RUN = True            # mude para False para gravar
ORIGIN_SHEET_ID = os.getenv('ORIGIN_SHEET_ID')   # modeloGeral (existente)
DEST_SHEET_ID   = '1DpH5tu4KJKqbA6ueFtf1s1FueBkR4-EtPf5xHyXx8zw'
ORIGIN_TAB      = 'modeloGeral'
DEST_TAB        = 'IMPULSIONAMENTOS 2025'
HEAD_ROW_DEST   = 5
MIN_DATE        = dt.date(2025, 6, 1)
assert ORIGIN_SHEET_ID, "Defina a variavel de ambiente ORIGIN_SHEET_ID"

DEST_COLUMNS = [
    'Data',
    'Campanha',
    'Veículo',
    'Link conteúdos impulsionados',
    'Período',
    'Agência',
    'Editoria',
    'Objetivo (aumentar seguidores, melhorar engajamento, etc)',
    'Meta (número) (quantos seguidores, compartilhamentos, etc. previstos?)',
    'Status',
    'Resultado',
]

In [ ]:
def normalize_vehicle(val):
    if not val:
        return ''
    v = str(val).strip().casefold()
    if v.startswith('facebook'):
        return 'FB'
    if v.startswith('instagram'):
        return 'IG'
    return str(val).strip()


def concat_period(start, end):
    if not start or not end:
        return ''
    s = normalize_date_to_str_DD_M_YYYY(start)
    e = normalize_date_to_str_DD_M_YYYY(end)
    return f'{s} a {e}'


def make_id(row):
    parts = [
        row['Data'],
        row['Campanha'],
        row['Veículo'],
        row['Link conteúdos impulsionados'],
        row['Período'],
        row['Agência'],
        row['Editoria'],
        row['Objetivo (aumentar seguidores, melhorar engajamento, etc)'],
    ]
    return '|'.join(str(p) for p in parts)


def transform_dataframe(df_origin):
    df = df_origin.copy()
    df['Data_dt'] = pd.to_datetime(df['date'], errors='coerce').dt.date
    df['Data'] = df['Data_dt'].apply(normalize_date_to_str_DD_M_YYYY)
    df['Campanha'] = df['Campanha']
    vehicle_col = df.get('Veículo').combine_first(df.get('Veiculo'))
    df['Veículo'] = vehicle_col.apply(normalize_vehicle)
    df['Link conteúdos impulsionados'] = df['URL_do_Anuncio']
    df['Período'] = df.apply(lambda r: concat_period(r.get('start'), r.get('end')), axis=1)
    df['Agência'] = 'De Brito'
    df['Editoria'] = df['Campanha']
    df['Objetivo (aumentar seguidores, melhorar engajamento, etc)'] = df['objective']
    df['Meta (número) (quantos seguidores, compartilhamentos, etc. previstos?)'] = ''
    df['Status'] = ''
    df['Resultado'] = ''
    cols = DEST_COLUMNS + ['Data_dt']
    return df[cols]

In [ ]:
df_origin = read_df(
    sheet_id=ORIGIN_SHEET_ID,
    tab=ORIGIN_TAB,
    header_row=0,
)
df_transf = transform_dataframe(df_origin)


In [ ]:
df_transf = df_transf[df_transf['Data_dt'] >= MIN_DATE]
df_transf.drop(columns=['Data_dt'], inplace=True)


In [ ]:
df_transf['__ID__'] = df_transf.apply(make_id, axis=1)


In [ ]:
df_dest = read_df(
    sheet_id=DEST_SHEET_ID,
    tab=DEST_TAB,
    header_row=HEAD_ROW_DEST - 1,
)
df_dest['__ID__'] = df_dest.apply(make_id, axis=1)

In [ ]:
novos = df_transf[~df_transf['__ID__'].isin(df_dest['__ID__'])]
print(f"{len(novos)} linhas novas após deduplicação.")


In [ ]:
df_final = novos.reindex(columns=DEST_COLUMNS, fill_value='').drop(columns=['__ID__'], errors='ignore')
if DRY_RUN:
    display(df_final.head())
    print(f'Novas linhas (deduplicadas): {len(df_final)}')
else:
    if not df_final.empty:
        write_dataframe_to_sheet(
            DEST_SHEET_ID,
            DEST_TAB,
            df_final,
            start_row=HEAD_ROW_DEST + 1,
            include_header=False,
        )
        print('Escrita concluída.')
    else:
        print('Nenhuma linha nova para escrever.')

Esta etapa gera um controle de novos impulsionamentos. Ajuste os parâmetros para publicar.